In [ ]:
# 0. Ansätze, die in diesem Notebook implementiert sind (inklusive Hinweis im Code, wo genau).

# (1) Batch-Normalisierung
# Aktivierbar über den Flag use_batchnorm. Laut vielen GNN-Papers sorgt BatchNorm dafür, dass die Verteilungen der Knoten-Features zwischen den Layern stabiler bleiben und das Training schneller konvergiert.
# Literatur: Ioffe, S. & Szegedy, C. (2015). Batch Normalization: Accelerating Deep Network Training by Reducing Internal Covariate Shift. In Proceedings of the 32nd International Conference on Machine Learning (ICML).

# (2) Dropout
# Optionales Dropout nach jeder Graph-Convolution, gesteuert über den Parameter dropout. Dropout hilft, Overfitting zu reduzieren, gerade bei kleinen Graph-Datasets.
# Literatur: Srivastava, N., Hinton, G., Krizhevsky, A., Sutskever, I. & Salakhutdinov, R. (2014). Dropout: A Simple Way to Prevent Neural Networks from Overfitting. Journal of Machine Learning Research, 15(56), 1929–1958.

# (3) Variierende Aktivierungsfunktionen
# Wir erlauben sowohl relu als auch selu als Aktivierung — beides Standard in GNN-Architekturen und empfohlen in verschiedenen Arbeiten, um non-linearities besser auszunutzen.
# Literatur: Klambauer, G., Unterthiner, T., Mayr, A. & Hochreiter, S. (2017). Self-Normalizing Neural Networks. In Advances in Neural Information Processing Systems (NeurIPS), 30.

# (4) Anzahl der Schichten und Hidden-Dimension
# num_layers und hidden_dim sind frei wählbar. Viele Papers zeigen, dass zu flache Netze underfitten, zu tiefe Netze aber auch Probleme machen können (“over-smoothing”). Mit unserer Grid-Search erkunden wir daher 2–4 Schichten und 32–128 Neuronen pro Layer.
# Literatur: Li, Q., Han, Z. & Wu, X.-M. (2018). Deeper Insights into Graph Convolutional Networks for Semi-Supervised Learning. In Proceedings of the AAAI Conference on Artificial Intelligence.

In [4]:
# 1. Installationsbefehle (einmal ausführen)
!pip install torch torch-geometric scikit-learn
# Optional: falls Fehler auftreten, nutze den PyG-Installations-Guide:
# !pip install torch-scatter torch-sparse torch-cluster torch-spline-conv torch-geometric -f https://data.pyg.org/whl/torch-<TORCH-VERSION>+<CUDA>.html

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.1/11.1 MB 22.1 MB/s eta 0:00:00 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 22.4/22.4 MB 24.4 MB/s eta 0:00:00a 0:00:01

[notice] A new release of pip is available: 25.0.1 -> 25.1.1
[notice] To update, run: pip install --upgrade pip


In [5]:
# 2. Imports und Setup
import torch
import torch.nn.functional as F
from torch_geometric.nn import GCNConv, global_mean_pool
from torch_geometric.data import DataLoader
from sklearn.model_selection import ParameterGrid

In [6]:
# 3. Modelldefinition
class SimpleGNN(torch.nn.Module):
    def __init__(self, in_channels, hidden_dim=64, num_layers=2,
                 activation='relu', dropout=0.5, use_batchnorm=False, num_classes=2):
        super().__init__()
        self.convs = torch.nn.ModuleList()
        # (1) Batch-Normalisierung
        self.batchnorms = torch.nn.ModuleList() if use_batchnorm else None
        # Input-Layer
        self.convs.append(GCNConv(in_channels, hidden_dim))
        if use_batchnorm:
            self.batchnorms.append(torch.nn.BatchNorm1d(hidden_dim))
        # Hidden-Layers
        for _ in range(num_layers - 1):
            self.convs.append(GCNConv(hidden_dim, hidden_dim))
            # (1) Batch-Normalisierung
            if use_batchnorm:
                self.batchnorms.append(torch.nn.BatchNorm1d(hidden_dim))
        self.dropout = dropout
        self.activation = getattr(F, activation)
        self.lin = torch.nn.Linear(hidden_dim, num_classes)

    def forward(self, x, edge_index, batch):
        for idx, conv in enumerate(self.convs):
            # (2) Dropout
            x = conv(x, edge_index)
            if self.batchnorms:
                x = self.batchnorms[idx](x)
            x = self.activation(x)
            # (2) Dropout
            x = F.dropout(x, p=self.dropout, training=self.training)
        x = global_mean_pool(x, batch)
        return self.lin(x)

In [7]:
# 4. Hyperparameter Grid
param_grid = {
    # (4) Anzahl der Schichten und Hidden-Dimension
    'hidden_dim': [32, 64, 128],
    'num_layers': [2, 3, 4],
    # (3) Variierende Aktivierungsfunktionen
    'activation': ['relu', 'selu'],
    'dropout': [0.0, 0.3, 0.6],
    'lr': [1e-3, 1e-4],
    'weight_decay': [0, 1e-5],
    'use_batchnorm': [False, True],
}

In [ ]:
# 5. Daten laden
# Beispiel: dataset = YourGraphDataset(root='data/')
# Aufteilen in Training und Validierung
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader   = DataLoader(val_dataset,   batch_size=32)

In [ ]:
# 5a. Feature Extraction
# Hier kannst du neue Knoten-Features berechnen, z.B. Knotengrade oder zentrale Maße:
import networkx as nx
from torch_geometric.utils import to_networkx, degree, add_self_loops

def extract_degree_features(data):
    # Konvertiere zu networkx-Graph
    G = to_networkx(data, to_undirected=True)
    # Berechne Grad-Feature
    deg = degree(data.edge_index[0], num_nodes=data.num_nodes)
    # Optional: füge als zusätzliche Dimension hinzu
    data.x = torch.cat([data.x, deg.unsqueeze(1)], dim=1)
    return data

# Anwenden als Transform auf Dataset
dataset.transform = extract_degree_features

In [ ]:
# 6. Trainings- und Evaluationsfunktion
def train_and_eval(config):
    model = SimpleGNN(
        in_channels=dataset.num_node_features,
        hidden_dim=config['hidden_dim'],
        num_layers=config['num_layers'],
        activation=config['activation'],
        dropout=config['dropout'],
        use_batchnorm=config['use_batchnorm'],
        num_classes=dataset.num_classes
    )
    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=config['lr'],
        weight_decay=config['weight_decay']
    )
    # Training
    for epoch in range(1, 51):
        model.train()
        for data in train_loader:
            optimizer.zero_grad()
            pred = model(data.x, data.edge_index, data.batch)
            loss = F.cross_entropy(pred, data.y)
            loss.backward()
            optimizer.step()
    # Validierung
    model.eval()
    correct = total = 0
    for data in val_loader:
        pred = model(data.x, data.edge_index, data.batch)
        _, labels = pred.max(dim=1)
        correct += (labels == data.y).sum().item()
        total += data.num_graphs
    return correct / total

In [ ]:
# 7. Grid-Search ausführen
results = []
for cfg in ParameterGrid(param_grid):
    acc = train_and_eval(cfg)
    results.append((cfg, acc))

In [ ]:
# 8. Bestes Ergebnis
best_cfg, best_acc = max(results, key=lambda x: x[1])
print("Best config:", best_cfg)
print("Validation accuracy:", best_acc)

In [ ]:
# 9. Nächste Schritte
# - Learning Rate Scheduler (ReduceLROnPlateau, CosineAnnealing)
# - Alternative Optimizer (AdamW, RAdam)
# - Andere GNN-Layer (GraphSAGE, GAT)
# - Feature-Engineering (Normierung, Augmentation)